# Prepare Disaster Places Data

This notebook prepares place-level building permit data for disaster analysis.

**Steps:**
1. Load BPS place-level monthly panel (21,011 places, 3.3M rows)
2. Assign CBSA codes and aggregate to metropolitan areas (406 metros)
3. **Select hurricane events** using FEMA IA damage threshold ($150M)
4. Select counties and BPS places for each qualifying hurricane
5. Add disaster-affected places as pseudo-metro entries to the panel
6. Document BPS survey regime changes affecting data coverage
7. Load FEMA disaster declarations and pull IA data from OpenFEMA API
8. Build disaster event dictionaries and group definitions
9. Save compact intermediate files for downstream analysis notebooks

**Selection criteria** (configurable in Cell 1):
- Hurricane inclusion: total FEMA IA housing damage >= $150M
- County display threshold: IA registration rate >= 3%
- Group 1 vs Group 2 split: county registration rate >= 15%

**Outputs:**
- `output/permits_panel.csv` — Monthly permits for 444 entries (406 metros + 38 places)
- `output/disaster_config.json` — Event definitions, IA summary, gap info
- `output/hurricane_place_selections.csv` — Hurricane-county-place selections with IA data

In [2]:
import pandas as pd
import numpy as np
import json
import requests
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

PROJECT_DIR = Path.cwd()
CONFIG_DIR = PROJECT_DIR / 'config'
OLD_DIR    = PROJECT_DIR / 'old'
OUTPUT_DIR = PROJECT_DIR / 'output'

# ══════════════════════════════════════════════════════════════════════
# USER PARAMETERS — adjust these to change event selection thresholds
# ══════════════════════════════════════════════════════════════════════
MIN_IA_DAMAGE_M = 150     # $M — minimum total IA housing damage to include a disaster
MIN_COUNTY_REG_PCT = 3    # % — minimum county reg rate to display in discovery output
EVENT_YEAR_MIN = 2005     # Discovery window start
EVENT_YEAR_MAX = 2024     # Discovery window end

CBSA_OVERRIDES = {
    '14600': '35840', '19380': '19430', '44600': '48260',
    '23020': '18880', '39140': '39150', '26180': '46520',
    '29140': '29200', '31100': '31080', '42060': '42200',
    '14060': '14010',
}

PERMIT_COLS = [
    'units_1_bldgs', 'units_1', 'units_1_value',
    'units_2_bldgs', 'units_2', 'units_2_value',
    'units_34_bldgs', 'units_34', 'units_34_value',
    'units_5plus_bldgs', 'units_5plus', 'units_5plus_value',
]

# State FIPS to abbreviation
STATE_FIPS = {
    '01':'AL','02':'AK','04':'AZ','05':'AR','06':'CA','08':'CO','09':'CT',
    '10':'DE','11':'DC','12':'FL','13':'GA','15':'HI','16':'ID','17':'IL',
    '18':'IN','19':'IA','20':'KS','21':'KY','22':'LA','23':'ME','24':'MD',
    '25':'MA','26':'MI','27':'MN','28':'MS','29':'MO','30':'MT','31':'NE',
    '32':'NV','33':'NH','34':'NJ','35':'NM','36':'NY','37':'NC','38':'ND',
    '39':'OH','40':'OK','41':'OR','42':'PA','44':'RI','45':'SC','46':'SD',
    '47':'TN','48':'TX','49':'UT','50':'VT','51':'VA','53':'WA','54':'WV',
    '55':'WI','56':'WY','72':'PR','78':'VI',
}

print(f"Selection criteria: IA damage >= ${MIN_IA_DAMAGE_M}M, "
      f"county reg rate >= {MIN_COUNTY_REG_PCT}%")
print(f"Discovery window: {EVENT_YEAR_MIN}-{EVENT_YEAR_MAX}")

Selection criteria: IA damage >= $150M, county reg rate >= 3%
Discovery window: 2005-2024


## Load BPS Place Data

In [5]:
places = pd.read_csv(
    OUTPUT_DIR / 'bps_place_monthly_panel.csv',
    dtype={'place_id': str, 'state_code': str, 'cbsa_code': str, 'county_code': str}
)
places['date'] = pd.to_datetime(
    places['year'].astype(str) + '-' + places['month'].astype(str) + '-01')

print(f"{places.shape[0]:,} rows, {places['place_id'].nunique():,} places, "
      f"{places['date'].min():%Y-%m} to {places['date'].max():%Y-%m}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_10512\3867967238.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  places = pd.read_csv(


3,278,007 rows, 21,011 places, 2000-01 to 2025-10


## Hurricane Event Selection

Hurricanes are selected using FEMA Individual Assistance (IA) data:
1. **Disaster-level filter**: Include hurricanes with total IA housing damage >= $150M
2. **County-level analysis**: For qualifying hurricanes, compute county IA registration
   rates (registrations / county population)
3. **Place selection**: For counties above the display threshold, select a BPS place

Counties are split into two groups for analysis:
- **Group 1** (>= 15% registration rate): Severe direct impact
- **Group 2** (3-15% registration rate): Moderate/indirect impact

In [17]:
# ── Step 1: Get hurricane disaster declarations ──
fema_disasters = pd.read_csv(CONFIG_DIR / 'fema_disasters_by_metro.csv')
fema_disasters['begin_date'] = pd.to_datetime(fema_disasters['begin_date'])

hurr_rows = fema_disasters[
    (fema_disasters['incident_type'] == 'Hurricane') &
    (fema_disasters['begin_date'].dt.year >= EVENT_YEAR_MIN) &
    (fema_disasters['begin_date'].dt.year <= EVENT_YEAR_MAX)]
hurr_dns = sorted(hurr_rows['disasterNumber'].unique())
dn_titles = hurr_rows.groupby('disasterNumber')['title'].first().to_dict()
dn_dates = hurr_rows.groupby('disasterNumber')['begin_date'].first().to_dict()

print(f"Hurricane declarations {EVENT_YEAR_MIN}-{EVENT_YEAR_MAX}: {len(hurr_dns)} unique DNs")

# ── Step 2: Pull IA data for all hurricane DNs (concurrent) ──
url_ia = "https://www.fema.gov/api/open/v2/HousingAssistanceOwners"

def pull_disaster_ia(dn):
    recs, skip = [], 0
    while True:
        params = {
            '$filter': f"disasterNumber eq {dn}",
            '$select': 'disasterNumber,state,county,validRegistrations,totalDamage',
            '$top': 1000, '$skip': skip
        }
        try:
            batch = requests.get(url_ia, params=params, timeout=120).json().get('HousingAssistanceOwners', [])
        except Exception:
            break
        if not batch:
            break
        recs.extend(batch)
        skip += len(batch)
    if not recs:
        return dn, 0, pd.DataFrame()
    df = pd.DataFrame(recs)
    total = df['totalDamage'].sum()
    county = df.groupby(['disasterNumber', 'state', 'county']).agg(
        registrations=('validRegistrations', 'sum'),
        damage=('totalDamage', 'sum')).reset_index()
    return dn, total, county

print(f"Pulling IA data for {len(hurr_dns)} hurricane DNs from FEMA API...")
ia_results = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(pull_disaster_ia, int(dn)): dn for dn in hurr_dns}
    done = 0
    for f in as_completed(futures):
        dn, total, county_df = f.result()
        ia_results[dn] = (total, county_df)
        done += 1
        if done % 20 == 0:
            print(f"  {done}/{len(hurr_dns)} complete...")
print(f"  {done}/{len(hurr_dns)} complete")

# ── Step 2.5: Group DNs by hurricane name ──
def normalize_hurricane_name(title):
    """'HURRICANE KATRINA' -> 'Katrina', 'REMNANTS OF HURRICANE IDA' -> 'Ida'"""
    t = title.strip().upper()
    t = t.replace('REMNANTS OF ', '').replace('HURRICANE ', '')
    return t.strip().title()

hurr_name_map = {}   # dn -> normalized name
hurr_groups = {}     # norm_name -> sorted list of DNs
for dn in hurr_dns:
    norm = normalize_hurricane_name(dn_titles.get(dn, '?'))
    hurr_name_map[dn] = norm
    hurr_groups.setdefault(norm, []).append(dn)
for k in hurr_groups:
    hurr_groups[k] = sorted(hurr_groups[k])

# State per DN (from IA county data)
dn_states = {}
for dn, (_, county_df) in ia_results.items():
    if len(county_df) > 0:
        dn_states[dn] = county_df['state'].iloc[0]

# Per-hurricane totals (sum across all DNs)
hurr_totals = {}
for name, dns in hurr_groups.items():
    hurr_totals[name] = sum(ia_results.get(dn, (0, None))[0] for dn in dns)

print(f"\nGrouped {len(hurr_dns)} DNs into {len(hurr_groups)} named hurricanes")

# ── Step 3: Filter by damage threshold (at hurricane level) ──
qualifying_names = sorted(
    [n for n, t in hurr_totals.items() if t >= MIN_IA_DAMAGE_M * 1e6],
    key=lambda n: hurr_totals[n], reverse=True)
qualifying_dns = sorted([dn for name in qualifying_names
                         for dn in hurr_groups[name]
                         if ia_results.get(dn, (0,))[0] > 0])




print(f"\n{'='*90}")
print(f"QUALIFYING HURRICANES (combined IA housing damage >= ${MIN_IA_DAMAGE_M}M)")
print(f"{'='*90}")
print(f"{'Hurricane':<12s} {'Year':<6s} {'DNs (state, damage)':<50s} {'Combined':>12s}")
print(f"{'-'*12} {'-'*6} {'-'*50} {'-'*12}")

for name in sorted(hurr_groups.keys(), key=lambda n: -hurr_totals.get(n, 0)):
    total = hurr_totals[name]
    if total == 0:
        continue
    dns = hurr_groups[name]
    dates = [dn_dates.get(dn) for dn in dns if pd.notna(dn_dates.get(dn))]
    yr = min(d.year for d in dates) if dates else '?'
    dn_parts = []
    for dn in sorted(dns):
        dt = ia_results.get(dn, (0, None))[0]
        if dt > 0:
            dn_parts.append(f"{dn}({dn_states.get(dn,'?')},${dt/1e6:,.0f}M)")
    dn_str = ' '.join(dn_parts)
    mark = ' <--' if name in qualifying_names else ''
    print(f"{name:<12s} {yr:<6}  {dn_str:<50s} ${total/1e6:>10,.0f}M{mark}")

print(f"\n{len(qualifying_names)} of {len([n for n,t in hurr_totals.items() if t > 0])} "
      f"hurricanes qualify (>= ${MIN_IA_DAMAGE_M}M), comprising {len(qualifying_dns)} DNs")

# ── Step 4: County FIPS from declarations API ──
url_decl = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
print(f"\nPulling county FIPS for {len(qualifying_dns)} qualifying DNs...")

def _pull_pages(url, params, key, delay=0.15):
    recs, skip = [], 0
    while True:
        params['$skip'] = skip
        try:
            batch = requests.get(url, params=params, timeout=120).json().get(key, [])
        except Exception:
            break
        if not batch:
            break
        recs.extend(batch)
        skip += len(batch)
        time.sleep(delay)
    return recs

all_hurr_decl = []
for dn in qualifying_dns:
    recs = _pull_pages(url_decl, {
        '$filter': f"disasterNumber eq {dn}",
        '$select': 'disasterNumber,state,fipsStateCode,fipsCountyCode,designatedArea',
        '$top': 1000}, 'DisasterDeclarationsSummaries', delay=0.1)
    all_hurr_decl.extend(recs)
hurr_decl_df = pd.DataFrame(all_hurr_decl)
hurr_decl_df['county_fips'] = hurr_decl_df['fipsStateCode'] + hurr_decl_df['fipsCountyCode']

def _clean_county(s):
    return (s.str.replace(r'\s*\(County\)', ' County', regex=True)
             .str.replace(r'\s*\(Parish\)', ' Parish', regex=True)
             .str.replace(r'\s*\(Borough\)', ' Borough', regex=True)
             .str.replace(r'\s*\(Census Area\)', ' Census Area', regex=True)
             .str.replace(r'\s*\(city\)', ' city', regex=True)
             .str.replace(r'\s*\(City\)', ' City', regex=True)
             .str.replace(r'\s*\(Municipality\)', ' Municipality', regex=True)
             .str.strip())

hurr_decl_df['county_clean'] = _clean_county(hurr_decl_df['designatedArea'])
hurr_fips_lookup = (hurr_decl_df[['state', 'county_clean', 'county_fips']]
                    .drop_duplicates()
                    .set_index(['state', 'county_clean'])['county_fips'].to_dict())

# ── Step 5: County populations (Census 2020) ──
print("Loading county populations (Census 2020)...")
try:
    pop_resp = requests.get(
        "https://api.census.gov/data/2020/dec/pl?get=P1_001N,NAME&for=county:*",
        timeout=60)
    pop_data = pop_resp.json()
    county_pops = {}
    for row in pop_data[1:]:
        cfips = row[2] + row[3]
        county_pops[cfips] = int(row[0])
    print(f"  {len(county_pops)} counties loaded")
except Exception as e:
    print(f"  Census API failed ({e}), estimating from BPS place populations")
    _pi = places.groupby(['state_code', 'county_code']).agg(pop=('pop', 'max')).reset_index()
    _pi['cfips'] = _pi['state_code'].str.zfill(2) + _pi['county_code'].str.zfill(3)
    county_pops = _pi.set_index('cfips')['pop'].to_dict()

# ── Step 6: Build BPS place lookup per county ──
place_info = (places.groupby('place_id')
              .agg(place_name=('place_name', 'first'),
                   state_code=('state_code', 'first'),
                   county_code=('county_code', 'first'),
                   pop=('pop', 'last'),
                   n_months=('year', 'count'),
                   avg_permits=('units_1', 'mean'))
              .reset_index())
place_info['county_fips'] = (place_info['state_code'].str.zfill(2)
                             + place_info['county_code'].str.zfill(3))

# ── Step 7: Build county IA and display by hurricane ──
hurr_county_all = pd.concat(
    [c for dn in qualifying_dns for _, c in [ia_results[dn]] if len(c) > 0],
    ignore_index=True)
hurr_county_all['county_clean'] = _clean_county(hurr_county_all['county'])
hurr_county_all['county_fips'] = hurr_county_all.apply(
    lambda r: hurr_fips_lookup.get((r['state'], r['county_clean'])), axis=1)
hurr_county_all['county_pop'] = hurr_county_all['county_fips'].map(county_pops)
hurr_county_all['reg_pct'] = (hurr_county_all['registrations']
                              / hurr_county_all['county_pop'] * 100)
hurr_county_all['damage_M'] = hurr_county_all['damage'] / 1e6
hurr_county_all['title'] = hurr_county_all['disasterNumber'].map(dn_titles)
hurr_county_all['hurricane'] = hurr_county_all['disasterNumber'].map(hurr_name_map)

print(f"\n{'='*100}")
print(f"COUNTY BREAKDOWNS BY HURRICANE (registration rate >= {MIN_COUNTY_REG_PCT}%)")
print(f"{'='*100}")

qualifying_hurricanes_rows = []
for name in qualifying_names:
    dns = hurr_groups[name]
    total = hurr_totals[name]
    dates = [dn_dates.get(dn) for dn in dns if pd.notna(dn_dates.get(dn))]
    earliest = min(dates) if dates else None

    # Build per-DN rows for downstream use
    for dn in dns:
        dn_total = ia_results.get(dn, (0, None))[0]
        if dn_total > 0:
            qualifying_hurricanes_rows.append({
                'dn': dn, 'state': dn_states.get(dn, '?'),
                'name': dn_titles.get(dn, '?'),
                'hurricane': name,
                'year': dn_dates.get(dn).year if pd.notna(dn_dates.get(dn)) else None,
                'dn_damage_M': dn_total / 1e6,
                'total_damage_M': total / 1e6})

    # Header
    dn_detail = ', '.join(
        f'DR-{dn} ({dn_states.get(dn,"?")}, ${ia_results[dn][0]/1e6:,.0f}M)'
        for dn in dns if ia_results.get(dn, (0,))[0] > 0)
    states = sorted(set(dn_states.get(dn, '?') for dn in dns))

    print(f"\n{'─'*90}")
    print(f"  {name} ({', '.join(states)}, {earliest.year if earliest else '?'}) "
          f"— Combined IA: ${total/1e6:,.0f}M")
    print(f"  DNs: {dn_detail}")

    # Counties from ALL DNs for this hurricane
    dn_counties = hurr_county_all[hurr_county_all['disasterNumber'].isin(dns)].copy()

    high = dn_counties[dn_counties['reg_pct'] >= MIN_COUNTY_REG_PCT].sort_values(
        'reg_pct', ascending=False)
    below = dn_counties[(dn_counties['reg_pct'] < MIN_COUNTY_REG_PCT) &
                        (dn_counties['reg_pct'] > 0)]

    if len(high) == 0:
        print(f"  No counties with >= {MIN_COUNTY_REG_PCT}% registration rate")
    else:
        print(f"  {'County':<30s} {'St':<4s} {'DN':<6s} {'Regs':>8s} {'Damage':>10s} {'Reg%':>7s}")
        print(f"  {'-'*30} {'-'*4} {'-'*6} {'-'*8} {'-'*10} {'-'*7}")
        for _, cr in high.iterrows():
            cfips = cr['county_fips'] if pd.notna(cr['county_fips']) else '?'
            dn = int(cr['disasterNumber'])
            print(f"  {cr['county_clean']:<30s} {cr['state']:<4s} {dn:<6d} "
                  f"{cr['registrations']:>8,.0f} ${cr['damage_M']:>8,.1f}M {cr['reg_pct']:>6.1f}%")
            if pd.notna(cr['county_fips']):
                cp = place_info[place_info['county_fips'] == cr['county_fips']]
                cp = cp.sort_values('avg_permits', ascending=False)
                for _, p in cp.iterrows():
                    print(f"      {p['place_id']}  {p['place_name']:<40s} "
                          f"pop={p['pop']:>8,.0f}  avg={p['avg_permits']:>5.1f}/mo  "
                          f"n={p['n_months']:>3d}mo")
    if len(below) > 0:
        print(f"  ({len(below)} additional counties with <{MIN_COUNTY_REG_PCT}% not shown)")

qualifying_hurricanes = pd.DataFrame(qualifying_hurricanes_rows)
print(f"\n{len(qualifying_names)} qualifying hurricanes, "
      f"{len(qualifying_hurricanes)} DNs across "
      f"{len(set(qualifying_hurricanes['state']))} states")

Hurricane declarations 2005-2024: 110 unique DNs
Pulling IA data for 110 hurricane DNs from FEMA API...
  20/110 complete...
  40/110 complete...
  60/110 complete...
  80/110 complete...
  100/110 complete...
  110/110 complete

Grouped 110 DNs into 45 named hurricanes

QUALIFYING HURRICANES (combined IA housing damage >= $150M)
Hurricane    Year   DNs (state, damage)                                    Combined
------------ ------ -------------------------------------------------- ------------
Katrina      2005    1603(FL,$4,146M) 1604(AL,$2,196M) 1605(AL,$122M)   $     6,465M <--
Sandy        2012    4085(NY,$1,472M) 4086(NJ,$706M) 4089(RI,$2M) 4091(MD,$4M) $     2,184M <--
Harvey       2017    4332(TX,$2,182M)                                   $     2,182M <--
Ian          2022    4673(FL,$1,666M) 4677(SC,$3M)                      $     1,669M <--
Ida          2021    4611(LA,$734M) 4614(NJ,$277M) 4615(NY,$210M) 4618(PA,$89M) 4626(MS,$10M) $     1,320M <--
Helene       2024    4828(

## Non-Hurricane Disaster Discovery

Rank wildfires, tornadoes, and earthquakes by total FEMA IA housing damage
to inform threshold selection. Places for these disaster types are set
manually in `nonhurr_place_selections.csv`.

In [19]:
# ── Discover top non-hurricane disasters by IA damage ──
# Uses _pull_pages and pull_disaster_ia from the hurricane cell above

discovery_types = [
    ('Fire', 10),
    ('Tornado', 10),
    ('Earthquake', 10),
]

for incident_type, top_n in discovery_types:
    # Get all declarations of this type in the event window
    decl_recs = _pull_pages(url_decl, {
        '$filter': f"incidentType eq '{incident_type}'"
                   f" and fyDeclared ge {EVENT_YEAR_MIN}"
                   f" and fyDeclared le {EVENT_YEAR_MAX}",
        '$select': 'disasterNumber,declarationTitle,incidentBeginDate,state',
        '$top': 1000,
    }, 'DisasterDeclarationsSummaries')

    if not decl_recs:
        print(f"\n{incident_type}: No declarations found")
        continue

    decl_df = pd.DataFrame(decl_recs)
    decl_df['begin_date'] = pd.to_datetime(decl_df['incidentBeginDate'])
    dns = sorted(decl_df['disasterNumber'].unique())
    dn_names = decl_df.groupby('disasterNumber')['declarationTitle'].first().to_dict()
    dn_dates = decl_df.groupby('disasterNumber')['begin_date'].first().to_dict()
    dn_states = decl_df.groupby('disasterNumber')['state'].apply(set).to_dict()

    print(f"\n{'─'*80}")
    print(f"{incident_type.upper()}S: {len(dns)} unique DNs — pulling IA data...")

    # Pull IA totals (concurrent)
    ia_totals = {}
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {pool.submit(pull_disaster_ia, int(dn)): dn for dn in dns}
        done = 0
        for f in as_completed(futures):
            dn, total, _ = f.result()
            ia_totals[dn] = total
            done += 1
            if done % 20 == 0:
                print(f"  {done}/{len(dns)} complete...")
    print(f"  {done}/{len(dns)} complete")

    # Group DNs by title (same event may span multiple states)
    disaster_summary = {}
    for dn in dns:
        title = dn_names.get(dn, '?')
        disaster_summary.setdefault(title, {
            'total_damage': 0, 'dns': [],
            'date': dn_dates.get(dn), 'states': set()})
        disaster_summary[title]['total_damage'] += ia_totals.get(dn, 0)
        disaster_summary[title]['dns'].append(dn)
        disaster_summary[title]['states'].update(dn_states.get(dn, set()))

    # Sort by total damage and display
    ranked = sorted(disaster_summary.items(),
                    key=lambda x: x[1]['total_damage'], reverse=True)

    print(f"\nTop {top_n} {incident_type}s by IA Total Damage "
          f"({EVENT_YEAR_MIN}-{EVENT_YEAR_MAX}):\n")
    print(f"{'#':>3}  {'Disaster':<45s}  {'Date':>10s}  "
          f"{'States':<12s}  {'IA Damage':>12s}  {'DNs'}")
    print(f"{'─'*3}  {'─'*45}  {'─'*10}  {'─'*12}  {'─'*12}  {'─'*15}")
    for i, (title, info) in enumerate(ranked[:top_n], 1):
        date_str = info['date'].strftime('%Y-%m-%d') if pd.notna(info['date']) else '?'
        states_str = ', '.join(sorted(info['states']))
        if len(states_str) > 12:
            states_str = states_str[:10] + '..'
        print(f"{i:>3}  {title:<45s}  {date_str:>10s}  "
              f"{states_str:<12s}  ${info['total_damage']/1e6:>10,.0f}M  "
              f"{info['dns']}")

    # Also show a few below the cutoff for context
    if len(ranked) > top_n:
        print(f"  ...")
        for i, (title, info) in enumerate(ranked[top_n:top_n+3], top_n+1):
            date_str = info['date'].strftime('%Y-%m-%d') if pd.notna(info['date']) else '?'
            print(f"{i:>3}  {title:<45s}  {date_str:>10s}  "
                  f"{'':12s}  ${info['total_damage']/1e6:>10,.0f}M")



────────────────────────────────────────────────────────────────────────────────
FIRES: 1033 unique DNs — pulling IA data...
  20/1033 complete...
  40/1033 complete...
  60/1033 complete...
  80/1033 complete...
  100/1033 complete...
  120/1033 complete...
  140/1033 complete...
  160/1033 complete...
  180/1033 complete...
  200/1033 complete...
  220/1033 complete...
  240/1033 complete...
  260/1033 complete...
  280/1033 complete...
  300/1033 complete...
  320/1033 complete...
  340/1033 complete...
  360/1033 complete...
  380/1033 complete...
  400/1033 complete...
  420/1033 complete...
  440/1033 complete...
  460/1033 complete...
  480/1033 complete...
  500/1033 complete...
  520/1033 complete...
  540/1033 complete...
  560/1033 complete...
  580/1033 complete...
  600/1033 complete...
  620/1033 complete...
  640/1033 complete...
  660/1033 complete...
  680/1033 complete...
  700/1033 complete...
  720/1033 complete...
  740/1033 complete...
  760/1033 complete...
  78

In [30]:
import re

# ======================================================================
# PLACE SELECTIONS — loaded from hurricane_places.csv + nonhurr_place_selections.csv
# ======================================================================

hurr_places_csv = pd.read_csv(OUTPUT_DIR / 'hurricane_places.csv',
                               dtype={'place_id': str})
nonhurr_csv = pd.read_csv(CONFIG_DIR / 'nonhurr_place_selections.csv',
                           dtype={'place_id': str})
hurr_places_csv['place_id'] = hurr_places_csv['place_id'].str.zfill(8)
nonhurr_csv['place_id'] = nonhurr_csv['place_id'].str.zfill(8)

print(f"Loaded {len(hurr_places_csv)} hurricane place selections "
      f"({hurr_places_csv['hurricane_name'].nunique()} hurricanes), "
      f"{len(nonhurr_csv)} non-hurricane selections")

def make_pseudo_name(bps_name, state_abbr):
    """Convert BPS place name to compact pseudo-metro display name."""
    name = bps_name.strip()
    name = re.sub(r'\s+County\s+Unincorporated\s+Area$', ' Co. (unincorp.)', name)
    name = re.sub(r'(\s+Parish)\s+Unincorporated\s+Area$', r'\1 (unincorp.)', name)
    name = re.sub(r'\s+County$', ' Co.', name)
    name = re.sub(r'\s+township$', ' twp.', name)
    name = re.sub(r'\s+city$', ' (city)', name)
    name = re.sub(r'^San Buenaventura$', 'Ventura', name)
    name = re.sub(r'\s+town$', '', name)
    return f"{name}, {state_abbr}"

# ── Build PLACE_PSEUDOS from both CSVs ──
hurr_places_csv['state_abbr'] = hurr_places_csv['place_id'].str[:2].map(STATE_FIPS)
hurr_places_csv['pseudo_name'] = hurr_places_csv.apply(
    lambda r: make_pseudo_name(r['place_name'], r['state_abbr']), axis=1)

PLACE_PSEUDOS = {}
for _, r in hurr_places_csv.drop_duplicates('place_id').iterrows():
    PLACE_PSEUDOS[r['place_id']] = r['pseudo_name']
for _, r in nonhurr_csv.iterrows():
    PLACE_PSEUDOS[r['place_id']] = r['pseudo_name']
print(f"PLACE_PSEUDOS: {len(PLACE_PSEUDOS)} unique places")

# ── Hurricane-specific processing: match to FEMA disaster numbers ──
hurr_merged = hurr_places_csv.merge(
    place_info[['place_id', 'county_fips', 'pop']],
    on='place_id', how='left')

def find_dn(hurricane_name, county_fips, state_code):
    """Find the DN for this hurricane covering this county/state."""
    norm = hurricane_name.strip().title()
    dns = hurr_groups.get(norm, [])
    if not dns:
        return None
    for dn in dns:
        if len(hurr_decl_df[(hurr_decl_df['disasterNumber'] == dn) &
                            (hurr_decl_df['county_fips'] == county_fips)]) > 0:
            return dn
    for dn in dns:
        if len(hurr_decl_df[(hurr_decl_df['disasterNumber'] == dn) &
                            (hurr_decl_df['fipsStateCode'] == state_code)]) > 0:
            return dn
    return dns[0] if dns else None

hurr_merged['dn'] = hurr_merged.apply(
    lambda r: find_dn(r['hurricane_name'], r.get('county_fips', ''),
                      r['place_id'][:2])
    if pd.notna(r.get('county_fips')) else None, axis=1)

no_dn = hurr_merged[hurr_merged['dn'].isna()]
if len(no_dn) > 0:
    print(f"\n  WARNING: {len(no_dn)} hurricane place(s) could not match a DN:")
    for _, r in no_dn.iterrows():
        print(f"    {r['hurricane_name']}: {r['pseudo_name']}")

# ── Build hurr_sel DataFrame ──
hurr_sel_rows = []
for _, r in hurr_merged.iterrows():
    if pd.isna(r['dn']) or pd.isna(r.get('county_fips')):
        continue
    dn = int(r['dn'])
    cfips = r['county_fips']
    cr = hurr_county_all[
        (hurr_county_all['disasterNumber'] == dn) &
        (hurr_county_all['county_fips'] == cfips)]
    if len(cr) > 0:
        c = cr.iloc[0]
        hurr_sel_rows.append({
            'hurricane': c.get('title', dn_titles.get(dn, '?')),
            'hurricane_name': r['hurricane_name'],
            'dn': dn, 'state': r['state_abbr'],
            'county': r['county_name'], 'county_fips': cfips,
            'county_reg': c['registrations'],
            'county_dmg_M': c.get('damage_M', c.get('damage', 0) / 1e6),
            'county_pop': c['county_pop'],
            'county_reg_pct': r['county_reg_pct'],
            'place_id': r['place_id'], 'place_name': r['pseudo_name'],
            'place_permits_yr': r['avg_permits_month'] * 12,
            'place_pop': r['pop'] if pd.notna(r.get('pop')) else 0,
            'pick_rank': 1})
    else:
        hurr_sel_rows.append({
            'hurricane': dn_titles.get(dn, '?'),
            'hurricane_name': r['hurricane_name'],
            'dn': dn, 'state': r['state_abbr'],
            'county': r['county_name'], 'county_fips': cfips,
            'county_reg': 0, 'county_dmg_M': 0,
            'county_pop': county_pops.get(cfips, 0),
            'county_reg_pct': r['county_reg_pct'],
            'place_id': r['place_id'], 'place_name': r['pseudo_name'],
            'place_permits_yr': r['avg_permits_month'] * 12,
            'place_pop': r['pop'] if pd.notna(r.get('pop')) else 0,
            'pick_rank': 1})

hurr_sel = pd.DataFrame(hurr_sel_rows)
print(f"\nHurricane selections: {len(hurr_sel)} entries, "
      f"{hurr_sel['hurricane_name'].nunique()} hurricanes")

# ── Non-hurricane events ──
nonhurr_events = nonhurr_csv.copy()
nonhurr_events['dn_csv'] = pd.to_numeric(nonhurr_events['dn_csv'], errors='coerce')
nonhurr_events['date_csv'] = pd.to_datetime(nonhurr_events['date_csv'], errors='coerce')
print(f"Non-hurricane events: {len(nonhurr_events)} entries")

# ── Data availability report ──
_place_first = places.groupby('place_id')['date'].min()
_place_last = places.groupby('place_id')['date'].max()

print(f"\n{'='*90}")
print(f"{'Type':<11s} {'Disaster':<15s} {'Place':<34s} {'Data range':<21s} {'Avg/mo':>7s}")
print(f"{'-'*11} {'-'*15} {'-'*34} {'-'*21} {'-'*7}")
for pid, pname in sorted(PLACE_PSEUDOS.items(), key=lambda x: x[1]):
    first = _place_first.get(pid)
    last = _place_last.get(pid)
    if first is None:
        print(f"{'?':<11s} {'?':<15s} {pname:<34s} {'NOT IN BPS':<21s}")
        continue
    # Find disaster info
    h = hurr_places_csv[hurr_places_csv['place_id'] == pid]
    n = nonhurr_csv[nonhurr_csv['place_id'] == pid]
    if len(h) > 0:
        dtype, dname = 'Hurricane', h.iloc[0]['hurricane_name']
        avg = h.iloc[0]['avg_permits_month']
    elif len(n) > 0:
        dtype, dname = n.iloc[0]['type'], n.iloc[0]['disaster']
        avg = n.iloc[0]['avg_permits']
    else:
        dtype, dname, avg = '?', '?', 0
    print(f"{dtype:<11s} {dname:<15s} {pname:<34s} "
          f"{first:%Y-%m} to {last:%Y-%m}   {avg:>6.1f}")


Loaded 399 hurricane place selections (19 hurricanes), 15 non-hurricane selections
PLACE_PSEUDOS: 269 unique places

    Tropical Storm Bonnie And Charley: Charlotte Co. (unincorp.), FL
    Tropical Storm Bonnie And Charley: Punta Gorda, FL
    Tropical Storm Bonnie And Charley: Hardee Co. (unincorp.), FL
    Tropical Storm Bonnie And Charley: Daytona Beach, FL
    Tropical Storm Bonnie And Charley: De Land, FL
    Tropical Storm Bonnie And Charley: Deltona, FL
    Tropical Storm Bonnie And Charley: Edgewater, FL
    Tropical Storm Bonnie And Charley: New Smyrna Beach, FL
    Tropical Storm Bonnie And Charley: Orange City, FL
    Tropical Storm Bonnie And Charley: Ormond Beach, FL
    Tropical Storm Bonnie And Charley: Port Orange, FL
    Tropical Storm Bonnie And Charley: Volusia Co. (unincorp.), FL
    Frances: Brevard Co. (unincorp.), FL
    Frances: Cocoa, FL
    Frances: Melbourne, FL
    Frances: Palm Bay, FL
    Frances: Rockledge, FL
    Frances: Satellite Beach, FL
    Frances

## CBSA Assignment and Panel Construction

In [33]:
# Backfill cbsa_code for 2000-2003 using mode from 2004+ data
valid_cbsa = places[(places['cbsa_code'].notna()) & (places['cbsa_code'] != '99999')]
place_cbsa_lookup = (valid_cbsa.groupby('place_id')['cbsa_code']
                     .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]))
mask_nan = places['cbsa_code'].isna()
places.loc[mask_nan, 'cbsa_code'] = places.loc[mask_nan, 'place_id'].map(place_cbsa_lookup)

# Apply CBSA code overrides
places['cbsa_code'] = places['cbsa_code'].map(
    lambda x: CBSA_OVERRIDES.get(x, x) if pd.notna(x) else x)

# Build cbsa_code -> metro_canonical from metro panel
metro_ref = pd.read_csv(CONFIG_DIR / 'bps_metro_monthly_panel.csv',
    usecols=['metro_canonical', 'cbsa'], dtype={'cbsa': str})
metro_ref['cbsa'] = metro_ref['cbsa'].map(
    lambda x: CBSA_OVERRIDES.get(x, x) if pd.notna(x) else x)
cbsa_to_metro = (metro_ref.dropna(subset=['cbsa'])
                 .drop_duplicates(subset=['cbsa'])
                 .set_index('cbsa')['metro_canonical'].to_dict())

places['metro_canonical'] = places['cbsa_code'].map(cbsa_to_metro)
places_metro = places[places['metro_canonical'].notna()].copy()
print(f"Mapped to {places_metro['metro_canonical'].nunique()} metros "
      f"({len(places_metro):,} rows)")

Mapped to 406 metros (2,373,250 rows)


In [35]:
# Aggregate places to metro-month level
panel = (places_metro
         .groupby(['metro_canonical', 'year', 'month', 'date'])[PERMIT_COLS]
         .sum().reset_index())
panel = panel.sort_values(['metro_canonical', 'date']).reset_index(drop=True)
print(f"Metro panel: {panel.shape[0]:,} rows, {panel['metro_canonical'].nunique()} metros")

# Remove CBSA metro rows that collide with pseudo-metro names
# (e.g., "Lake Charles, LA" exists as both a CBSA metro and a BPS place)
pseudo_names = set(PLACE_PSEUDOS.values())
collisions = set(panel['metro_canonical'].unique()) & pseudo_names
if collisions:
    print(f"Removing {len(collisions)} CBSA metro(s) that collide "
          f"with pseudo names: {', '.join(sorted(collisions))}")
    panel = panel[~panel['metro_canonical'].isin(collisions)]

# Add pseudo-metro entries for each selected place
print(f"\nAdding {len(PLACE_PSEUDOS)} place-level pseudo-metro entries:")
for place_id, pseudo_name in PLACE_PSEUDOS.items():
    pdata = places[places['place_id'] == place_id].copy()
    if len(pdata) == 0:
        print(f"  WARNING: {place_id} ({pseudo_name}) not found")
        continue
    pagg = (pdata.groupby(['year', 'month', 'date'])[PERMIT_COLS]
            .sum().reset_index())
    pagg['metro_canonical'] = pseudo_name
    panel = pd.concat([panel, pagg], ignore_index=True)
    avg = pagg['units_1'].mean()
    print(f"  {pseudo_name}: {len(pagg)} months, {avg:.1f} avg permits/mo")

# Mark which entries are pseudo-metros
pseudo_names = set(PLACE_PSEUDOS.values())
panel['is_pseudo'] = panel['metro_canonical'].isin(pseudo_names)
panel = panel.sort_values(['metro_canonical', 'date']).reset_index(drop=True)
print(f"\nFinal panel: {panel.shape[0]:,} rows, {panel['metro_canonical'].nunique()} entries "
      f"({panel['is_pseudo'].sum():,} pseudo-metro rows)")

Metro panel: 115,637 rows, 406 metros
Removing 9 CBSA metro(s) that collide with pseudo names: Alexandria, LA, Baton Rouge, LA, Hattiesburg, MS, Lake Charles, LA, Mobile, AL, Napa, CA, Ocean City, NJ, Port St. Lucie, FL, Punta Gorda, FL

Adding 269 place-level pseudo-metro entries:
  Charlotte Co. (unincorp.), FL: 310 months, 129.9 avg permits/mo
  Punta Gorda, FL: 226 months, 8.8 avg permits/mo
  Hardee Co. (unincorp.), FL: 226 months, 5.5 avg permits/mo
  Daytona Beach, FL: 310 months, 26.3 avg permits/mo
  De Land, FL: 310 months, 31.9 avg permits/mo
  Deltona, FL: 310 months, 34.4 avg permits/mo
  Edgewater, FL: 226 months, 10.8 avg permits/mo
  New Smyrna Beach, FL: 310 months, 16.8 avg permits/mo
  Orange City, FL: 226 months, 3.3 avg permits/mo
  Ormond Beach, FL: 226 months, 11.9 avg permits/mo
  Port Orange, FL: 310 months, 20.1 avg permits/mo
  Volusia Co. (unincorp.), FL: 226 months, 51.0 avg permits/mo
  Brevard Co. (unincorp.), FL: 310 months, 92.0 avg permits/mo
  Cocoa, 

## BPS Survey Regime Changes

The Census Bureau's Building Permits Survey underwent two major reorganizations
that affect place-level data coverage:

1. **2015 redesign**: ~2,600 places dropped from the survey, ~1,800 new places added
2. **2022 expansion**: ~11,500 places added (many previously dropped places restored)

This creates three populations among our 38 selected places:
- **Continuous** (data 2000-2025): survived both changes
- **Gap** (data 2000-2014 + 2022-2025): dropped in 2015, restored in 2022
- **Post-2015 only** (data 2015-2025): added in the 2015 redesign

Missing months represent places **absent from the survey** (not reporting at all),
which is distinct from months with zero permits (explicitly reported as zero).

In [37]:
# Detect survey gaps for each pseudo-metro place
gap_info = {}
for place_id, pseudo_name in PLACE_PSEUDOS.items():
    pdata = places[places['place_id'] == place_id]
    if len(pdata) == 0:
        gap_info[pseudo_name] = None
        continue
    dates = set(pd.to_datetime(
        pdata['year'].astype(str) + '-' + pdata['month'].astype(str) + '-01'))
    full_range = pd.date_range(
        min(dates), max(dates), freq='MS')
    missing = sorted(set(full_range) - dates)

    if len(missing) <= 2:  # just Nov/Dec 2025
        gap_info[pseudo_name] = None
        continue

    # Find contiguous gap periods (ignore the trailing 2025 partial year)
    gaps = []
    current_start = missing[0]
    current_end = missing[0]
    for i in range(1, len(missing)):
        if (missing[i] - current_end).days <= 35:
            current_end = missing[i]
        else:
            n = len(pd.date_range(current_start, current_end, freq='MS'))
            if n >= 6:  # only report substantial gaps
                gaps.append({'start': current_start.strftime('%Y-%m'),
                             'end': current_end.strftime('%Y-%m'),
                             'months': n})
            current_start = missing[i]
            current_end = missing[i]
    n = len(pd.date_range(current_start, current_end, freq='MS'))
    if n >= 6:
        gaps.append({'start': current_start.strftime('%Y-%m'),
                     'end': current_end.strftime('%Y-%m'),
                     'months': n})

    gap_info[pseudo_name] = gaps if gaps else None

# Print summary
print(f"{'Place':<38s} {'Coverage':<15s} {'Gap period':<25s} {'Gap months':>10s}")
print('-' * 90)
for name in sorted(gap_info.keys()):
    info = gap_info[name]
    pdata = places[places['place_id'] == {v: k for k, v in PLACE_PSEUDOS.items()}[name]]
    yr_min, yr_max = pdata['year'].min(), pdata['year'].max()
    if info is None:
        print(f"{name:<38s} {yr_min}-{yr_max:<10d} {'Continuous':<25s}")
    else:
        for g in info:
            print(f"{name:<38s} {yr_min}-{yr_max:<10d} "
                  f"{g['start']} to {g['end']:<15s} {g['months']:>10d}")
n_gapped = sum(1 for v in gap_info.values() if v is not None)
print(f"\n{n_gapped} of {len(gap_info)} places have survey gaps")

Place                                  Coverage        Gap period                Gap months
------------------------------------------------------------------------------------------
Aberdeen twp., NJ                      2000-2025       Continuous               
Acadia Parish (unincorp.), LA          2000-2025       2015-01 to 2021-12                 84
Addis, LA                              2000-2025       Continuous               
Alexandria, LA                         2000-2025       Continuous               
Alvin, TX                              2000-2025       Continuous               
Angleton, TX                           2000-2025       Continuous               
Ascension Parish (unincorp.), LA       2004-2025       Continuous               
Assumption Parish, LA                  2000-2025       2015-01 to 2021-12                 84
Atlantic City, NJ                      2000-2025       Continuous               
Avalon borough, NJ                     2000-2025       Continuou

## FEMA Disaster Data and IA Pulls

In [40]:
# Load pre-built FEMA disaster-metro mapping
disasters = pd.read_csv(CONFIG_DIR / 'fema_disasters_by_metro.csv')
for col in ['begin_date', 'end_date', 'declaration_date']:
    disasters[col] = pd.to_datetime(disasters[col], errors='coerce')
disaster_dates = disasters.groupby('disasterNumber')['begin_date'].first().to_dict()
print(f"FEMA disasters: {disasters['disasterNumber'].nunique()} unique, "
      f"{disasters['metro_canonical'].nunique()} metros")

FEMA disasters: 395 unique, 346 metros


In [41]:
# ── Reuse hurricane IA from Cell 5; pull fire & other-event IA fresh ──

# Hurricane IA: reuse hurr_county_all from Cell 5 (already has county-level data)
ia_county_raw = hurr_county_all[['disasterNumber', 'state', 'county',
                                  'registrations', 'damage']].copy()
ia_county_raw['name'] = ia_county_raw['disasterNumber'].map(dn_titles)
ia_county_raw['inspected'] = 0
print(f"Hurricane IA (from Cell 5): {len(ia_county_raw)} county records, "
      f"{ia_county_raw['disasterNumber'].nunique()} DNs across "
      f"{len(set(hurr_name_map[dn] for dn in ia_county_raw['disasterNumber'].unique()))} hurricanes")
for name in qualifying_names:
    dns = hurr_groups[name]
    sub = ia_county_raw[ia_county_raw['disasterNumber'].isin(dns)]
    dn_str = ', '.join(str(d) for d in sorted(dns))
    print(f"  {name:<12s} (DNs {dn_str}): {len(sub):>3d} counties, {sub['registrations'].sum():>10,} reg")

# Hurricane FIPS: reuse hurr_decl_df from Cell 5
decl_raw = hurr_decl_df.copy()
print(f"\nHurricane FIPS (from Cell 5): {len(decl_raw)} declaration records")

# Build fire and other-event lists from CSV selections (Cell 6)
_fire_csv = nonhurr_events[nonhurr_events['type'] == 'Fire']
ia_fires = list(_fire_csv.groupby(['dn_csv', 'state_abbr', 'disaster']).size()
    .reset_index().apply(lambda r: (int(r['dn_csv']), r['state_abbr'], r['disaster']), axis=1))
_other_csv = nonhurr_events[nonhurr_events['type'].isin(['Tornado', 'Earthquake'])]
ia_other = list(_other_csv.groupby(['dn_csv', 'state_abbr', 'disaster']).size()
    .reset_index().apply(lambda r: (int(r['dn_csv']), r['state_abbr'], r['disaster']), axis=1))

url_ia = "https://www.fema.gov/api/open/v2/HousingAssistanceOwners"
url_decl = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

def _pull_pages(url, params, key, delay=0.15, retries=3):
    recs, skip = [], 0
    while True:
        params['$skip'] = skip
        for attempt in range(retries):
            resp = requests.get(url, params=params, timeout=120)
            if resp.status_code == 200:
                try:
                    batch = resp.json().get(key, [])
                    break
                except Exception:
                    pass
            time.sleep(2 ** attempt)
        else:
            print(f"  WARNING: API failed after {retries} retries (status {resp.status_code})")
            break
        if not batch: break
        recs.extend(batch); skip += len(batch); time.sleep(delay)
    return recs

# Fire IA at state level
print("\nPulling fire IA (state-level)...")
fire_ia_raw = {}
for dn, st, name in ia_fires:
    recs = _pull_pages(url_ia, {
        '$filter': f"disasterNumber eq {dn} and state eq '{st}'",
        '$select': 'validRegistrations,totalInspected,totalDamage',
        '$top': 1000}, 'HousingAssistanceOwners')
    fire_ia_raw[dn] = {
        'registrations': sum(r.get('validRegistrations',0) or 0 for r in recs),
        'damage_usd': sum(r.get('totalDamage',0) or 0 for r in recs)}
    print(f"  DR-{dn} {name:<16s}: {fire_ia_raw[dn]['registrations']:>10,} reg, "
          f"${fire_ia_raw[dn]['damage_usd']/1e6:,.0f}M")

# Other events IA at county level
print("\nPulling other-event IA (county-level)...")
other_county_ia = []
for dn, st, name in ia_other:
    recs = _pull_pages(url_ia, {
        '$filter': f"disasterNumber eq {dn} and state eq '{st}'",
        '$select': 'disasterNumber,state,county,validRegistrations,totalInspected,totalDamage',
        '$top': 1000}, 'HousingAssistanceOwners')
    df = pd.DataFrame(recs)
    if df.empty:
        print(f"  DR-{dn} {name:<20s}:   0 counties,          0 reg (no IA data)")
        continue
    agg = df.groupby(['disasterNumber','state','county']).agg(
        registrations=('validRegistrations','sum'), inspected=('totalInspected','sum'),
        damage=('totalDamage','sum')).reset_index()
    agg['name'] = name
    other_county_ia.append(agg)
    print(f"  DR-{dn} {name:<20s}: {len(agg):>3d} counties, {agg['registrations'].sum():>10,} reg")
other_county_raw = pd.concat(other_county_ia, ignore_index=True)

# Other event FIPS
other_decl = []
for dn, st, _ in ia_other:
    recs = _pull_pages(url_decl, {
        '$filter': f"disasterNumber eq {dn} and state eq '{st}'",
        '$select': 'disasterNumber,state,fipsStateCode,fipsCountyCode,designatedArea',
        '$top': 1000}, 'DisasterDeclarationsSummaries', delay=0.1)
    other_decl.extend(recs)
other_decl_raw = pd.DataFrame(other_decl)

print(f"\nDone: {len(ia_county_raw)} hurricane county records, "
      f"{len(other_county_raw)} other county records")

Hurricane IA (from Cell 5): 1089 county records, 45 DNs across 19 hurricanes
  Katrina      (DNs 1602, 1603, 1604, 1605): 132 counties,    904,536 reg
  Sandy        (DNs 4085, 4086, 4089, 4090, 4091, 4092, 4093, 4095, 4096, 4097, 4099):  38 counties,    304,530 reg
  Harvey       (DNs 4332):  41 counties,    443,258 reg
  Ian          (DNs 4673, 4677):  31 counties,    488,608 reg
  Ida          (DNs 4611, 4614, 4615, 4618, 4626, 4627):  63 counties,    571,221 reg
  Helene       (DNs 4828, 4829, 4830): 124 counties,    655,058 reg
  Ike          (DNs 1791, 1792):  55 counties,    458,270 reg
  Rita         (DNs 1606, 1607):  67 counties,    446,094 reg
  Wilma        (DNs 1609):  14 counties,    413,253 reg
  Milton       (DNs 4834):  35 counties,    577,737 reg
  Irene        (DNs 4019, 4020, 4021, 4024, 4025, 4034, 4036, 4037):  99 counties,    170,376 reg
  Irma         (DNs 4337, 4338, 4346):  68 counties,  1,219,819 reg
  Beryl        (DNs 4798):  22 counties,    576,988 reg
  G

In [42]:
# Clean county names for FIPS matching
def _clean(s):
    return (s.str.replace(r'\s*\(County\)', ' County', regex=True)
             .str.replace(r'\s*\(Parish\)', ' Parish', regex=True)
             .str.replace(r'\s*\(Borough\)', ' Borough', regex=True)
             .str.replace(r'\s*\(Census Area\)', ' Census Area', regex=True)
             .str.replace(r'\s*\(city\)', ' city', regex=True)
             .str.replace(r'\s*\(City\)', ' City', regex=True)
             .str.replace(r'\s*\(Municipality\)', ' Municipality', regex=True)
             .str.strip())

ia_county_raw['county_clean'] = _clean(ia_county_raw['county'])
decl_raw['county_clean'] = _clean(decl_raw['designatedArea'])
decl_raw['county_fips'] = decl_raw['fipsStateCode'] + decl_raw['fipsCountyCode']
other_county_raw['county_clean'] = _clean(other_county_raw['county'])

# Combined FIPS lookup
all_decl_combined = pd.concat([decl_raw, other_decl_raw], ignore_index=True)
all_decl_combined['county_clean'] = _clean(all_decl_combined['designatedArea'])
all_decl_combined['county_fips'] = all_decl_combined['fipsStateCode'] + all_decl_combined['fipsCountyCode']
fips_lookup = (all_decl_combined[['state','county_clean','county_fips']]
               .drop_duplicates().set_index(['state','county_clean'])['county_fips'].to_dict())

ia_county_raw['county_fips'] = ia_county_raw.apply(
    lambda r: fips_lookup.get((r['state'], r['county_clean'])), axis=1)
other_county_raw['county_fips'] = other_county_raw.apply(
    lambda r: fips_lookup.get((r['state'], r['county_clean'])), axis=1)

# County FIPS -> CBSA -> metro
_pl = pd.read_csv(OUTPUT_DIR / 'bps_place_monthly_panel.csv',
    usecols=['state_code','county_code','cbsa_code'],
    dtype={'cbsa_code': str, 'state_code': str, 'county_code': str})
_valid = _pl[(_pl['cbsa_code'].notna()) & (_pl['cbsa_code'] != '99999')].copy()
_valid['cfips'] = _valid['state_code'].str.zfill(2) + _valid['county_code'].str.zfill(3)
_county_cbsa = _valid.groupby('cfips')['cbsa_code'].agg(
    lambda x: x.mode().iloc[0]).to_dict()
del _pl, _valid

_c2m = {k: v for k, v in cbsa_to_metro.items()}

ia_county_raw['cbsa_code'] = ia_county_raw['county_fips'].map(_county_cbsa)
ia_county_raw['cbsa_code'] = ia_county_raw['cbsa_code'].map(
    lambda x: CBSA_OVERRIDES.get(x,x) if pd.notna(x) else x)
ia_county_raw['metro_canonical'] = ia_county_raw['cbsa_code'].map(_c2m)

ia_metro = (ia_county_raw[ia_county_raw['metro_canonical'].notna()]
    .groupby(['disasterNumber','name','metro_canonical'])
    .agg(registrations=('registrations','sum'), damage=('damage','sum')).reset_index())
ia_metro['damage_M'] = ia_metro['damage'] / 1e6

# Build ia_summary: per-disaster IA for annotations
ia_summary = dict(fire_ia_raw)  # fires keyed by dn
for _, r in ia_metro.iterrows():
    ia_summary[(r['disasterNumber'], r['metro_canonical'])] = {
        'registrations': r['registrations'], 'damage_usd': r['damage']}

# Other events
other_county_raw['cbsa_code'] = other_county_raw['county_fips'].map(_county_cbsa)
other_county_raw['cbsa_code'] = other_county_raw['cbsa_code'].map(
    lambda x: CBSA_OVERRIDES.get(x,x) if pd.notna(x) else x)
other_county_raw['metro_canonical'] = other_county_raw['cbsa_code'].map(_c2m)

ia_other_metro = (other_county_raw[other_county_raw['metro_canonical'].notna()]
    .groupby(['disasterNumber','name','metro_canonical'])
    .agg(registrations=('registrations','sum'), damage=('damage','sum')).reset_index())
for _, r in ia_other_metro.iterrows():
    ia_summary[(r['disasterNumber'], r['metro_canonical'])] = {
        'registrations': r['registrations'], 'damage_usd': r['damage']}

# Place-level IA for fire/other events (from CSV county_fips)
_nonhurr_county_map = {}
for _, r in nonhurr_events.iterrows():
    if pd.notna(r.get('county_fips')) and pd.notna(r.get('dn_csv')):
        _nonhurr_county_map[(r['pseudo_name'], int(r['dn_csv']))] = r['county_fips']

print(f"ia_summary: {len(ia_summary)} entries")

ia_summary: 272 entries


## Event Definitions

In [44]:
# Build hurricane event groups from selections (hurr_sel built in Cell 6)
hurricane_events = {}

for _, r in hurr_sel.iterrows():
    pseudo_name = PLACE_PSEUDOS.get(r['place_id'])
    if not pseudo_name:
        continue
    dt = disaster_dates.get(r['dn'])
    if dt is None or (hasattr(dt, 'year') and pd.isna(dt)):
        continue
    event_entry = (pd.Timestamp(dt).strftime('%Y-%m-%d'), r['hurricane'], r['dn'])
    hurricane_events.setdefault(pseudo_name, []).append(event_entry)
    ia_summary[(r['dn'], pseudo_name)] = {
        'registrations': int(r['county_reg']),
        'damage_usd': r['county_dmg_M'] * 1e6,
        'reg_pct': r['county_reg_pct']}

for d in [hurricane_events]:
    for k in d:
        d[k].sort()

# Build fire and other event dicts from CSV selections (Cell 6)
fire_events = {}
tornado_events = {}
earthquake_events = {}
for _, r in nonhurr_events.iterrows():
    pname = r['pseudo_name']
    dn = int(r['dn_csv'])
    date_str = pd.Timestamp(r['date_csv']).strftime('%Y-%m-%d')
    disaster_name = r['disaster']
    event_entry = (date_str, disaster_name, dn)

    if r['type'] == 'Fire':
        fire_events.setdefault(pname, []).append(event_entry)
    elif r['type'] == 'Tornado':
        tornado_events.setdefault(pname, []).append(event_entry)
    elif r['type'] == 'Earthquake':
        earthquake_events.setdefault(pname, []).append(event_entry)

    # IA summary for fire/other places
    cfips = _nonhurr_county_map.get((pname, dn))
    if cfips:
        cdata = other_county_raw[
            (other_county_raw['county_fips'] == cfips) &
            (other_county_raw['disasterNumber'] == dn)]
        if len(cdata) > 0:
            cr = cdata.iloc[0]
            ia_summary[(dn, pname)] = {
                'registrations': int(cr['registrations']),
                'damage_usd': cr['damage']}

for d in [fire_events, tornado_events, earthquake_events]:
    for k in d:
        d[k].sort()

# Suppress combined Camp+Woolsey IA for Malibu's Woolsey entry if present
if ('Malibu, CA' in fire_events and
    any(e[2] == 4407 for e in fire_events.get('Malibu, CA', []))):
    ia_summary[(4407, 'Malibu, CA')] = {'registrations': 0, 'damage_usd': 0}

# Summary
all_events = {**fire_events, **hurricane_events, **tornado_events, **earthquake_events}
print(f"Event definitions:")
print(f"  Wildfires: {len(fire_events)} places, "
      f"{sum(len(v) for v in fire_events.values())} events")
print(f"  Hurricanes: {len(hurricane_events)} places, "
      f"{sum(len(v) for v in hurricane_events.values())} events")
print(f"  Tornadoes: {len(tornado_events)} places, "
      f"{sum(len(v) for v in tornado_events.values())} events")
print(f"  Earthquakes: {len(earthquake_events)} places, "
      f"{sum(len(v) for v in earthquake_events.values())} events")
print(f"  Total: {len(all_events)} unique places")

Event definitions:
  Wildfires: 11 places, 11 events
  Hurricanes: 228 places, 349 events
  Tornadoes: 2 places, 2 events
  Earthquakes: 2 places, 2 events
  Total: 243 unique places


## Save Intermediate Files

In [46]:
# Save panel (without date column -- reconstruct from year/month)
panel_save = panel.drop(columns=['date'])
panel_save.to_csv(OUTPUT_DIR / 'permits_panel.csv', index=False)
print(f"Saved permits_panel.csv: {len(panel_save):,} rows, "
      f"{panel_save['metro_canonical'].nunique()} entries")

# Save hurricane place selections
hurr_sel.to_csv(CONFIG_DIR / 'hurricane_place_selections.csv', index=False)
print(f"  hurricane_name column: {hurr_sel['hurricane_name'].nunique()} unique hurricanes")
print(f"Saved hurricane_place_selections.csv: {len(hurr_sel)} entries")

# Save non-hurricane selections summary
if len(nonhurr_events) > 0:
    nonhurr_save = nonhurr_events[['type', 'disaster', 'place_id', 'pseudo_name',
                                    'dn_csv', 'date_csv', 'state_abbr',
                                    'avg_permits', 'pop']].copy()
    nonhurr_save.to_csv(CONFIG_DIR / 'nonhurr_place_selections.csv', index=False)
    print(f"Saved nonhurr_place_selections.csv: {len(nonhurr_save)} entries "
          f"({nonhurr_save['type'].nunique()} types)")

# Build JSON-safe config
def _serialize_events(d):
    return {k: [list(e) for e in v] for k, v in d.items()}

def _serialize_ia(d):
    out = {}
    for key, val in d.items():
        if isinstance(key, tuple):
            str_key = f"{key[0]}|{key[1]}"
        else:
            str_key = str(key)
        out[str_key] = val
    return out

def _serialize_dates(d):
    out = {}
    for k, v in d.items():
        if hasattr(v, 'strftime'):
            out[str(k)] = v.strftime('%Y-%m-%d')
        else:
            out[str(k)] = str(v)
    return out

config = {
    'place_pseudos': PLACE_PSEUDOS,
    'fire_events': _serialize_events(fire_events),
    'hurricane_events': _serialize_events(hurricane_events),
    'tornado_events': _serialize_events(tornado_events),
    'earthquake_events': _serialize_events(earthquake_events),
    'ia_summary': _serialize_ia(ia_summary),
    'gap_info': gap_info,
    'disaster_dates': _serialize_dates(disaster_dates),
}

with open(OUTPUT_DIR / 'disaster_config.json', 'w') as f:
    json.dump(config, f, indent=2, default=str)

# Verify
with open(OUTPUT_DIR / 'disaster_config.json') as f:
    check = json.load(f)
print(f"Saved disaster_config.json: {len(check)} top-level keys")
print(f"  place_pseudos: {len(check['place_pseudos'])} places")
print(f"  ia_summary: {len(check['ia_summary'])} entries")
print(f"  gap_info: {sum(1 for v in check['gap_info'].values() if v)} places with gaps")

Saved permits_panel.csv: 191,036 rows, 665 entries
  hurricane_name column: 15 unique hurricanes
Saved hurricane_place_selections.csv: 349 entries
Saved nonhurr_place_selections.csv: 15 entries (3 types)
Saved disaster_config.json: 8 top-level keys
  place_pseudos: 269 places
  ia_summary: 614 entries
  gap_info: 37 places with gaps
